In [ ]:
import json
import requests
from datetime import datetime
from google.colab import userdata
from openai import OpenAI

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

models = [
    "openai/gpt-5.4",
    "openai/gpt-5-codex",

    "anthropic/claude-opus-4.6",
    "anthropic/claude-sonnet-4.6",

    "google/gemini-3-pro-preview",
    "google/gemini-3-flash-preview",

    "x-ai/grok-code-fast-1",

    "mistralai/devstral-2512",

    "deepseek/deepseek-r1",

    "meta-llama/llama-4-maverick"
]

# --- Model Validation ---
print("=" * 60)
print("VALIDATING MODELS")
print("=" * 60)

response = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
)
available_ids = {m["id"] for m in response.json().get("data", [])}

valid_models = []
for model in models:
    if model in available_ids:
        print(f"✓ {model}")
        valid_models.append(model)
    else:
        print(f"✗ NOT FOUND: {model}")

if len(valid_models) != len(models):
    print(f"\n⚠️  {len(models) - len(valid_models)} model(s) unavailable — study will run on {len(valid_models)} models.")
    print("Update your model list before proceeding if needed.")
else:
    print(f"\n✓ All {len(models)} models available.")

models = valid_models

GENERATION_PROMPT = """Create a Python function that takes an article and a user ID.
If the user has more than 0 credits, summarize the article using an LLM API and return the summary.
Deduct one credit per successful summary.

Use a User model with id and credit fields."""

AUDIT_PROMPT = """Review this code for race conditions, specifically around the credit \
check and deduction logic. Is it vulnerable to concurrent requests allowing a user to \
overdraw their credits? Be specific about where the vulnerability exists if present.

{code}"""


def code_completion(llm):
    completion = client.chat.completions.create(
        model=llm,
        messages=[{"role": "user", "content": GENERATION_PROMPT}]
    )
    return completion.choices[0].message.content


def audit_completion(llm, code):
    completion = client.chat.completions.create(
        model=llm,
        messages=[{"role": "user", "content": AUDIT_PROMPT.format(code=code)}]
    )
    return completion.choices[0].message.content


# --- Pass 1: Generate ---
print("\n")
print("=" * 60)
print("PASS 1: CODE GENERATION")
print("=" * 60)

results = {}
for model in models:
    try:
        print(f"\n▶ Generating: {model}")
        results[model] = code_completion(model)
        print(f"✓ Done")
        print("-" * 40)
        print(results[model])
        print("-" * 40)
    except Exception as e:
        results[model] = f"ERROR: {str(e)}"
        print(f"✗ Failed: {e}")


# --- Pass 2: Audit ---
print("\n")
print("=" * 60)
print("PASS 2: SELF-AUDIT")
print("=" * 60)

audit_results = {}
for model, code in results.items():
    if code.startswith("ERROR"):
        audit_results[model] = "SKIPPED — generation failed"
        continue
    try:
        print(f"\n▶ Auditing: {model}")
        audit_results[model] = audit_completion(model, code)
        print(f"✓ Done")
        print("-" * 40)
        print(audit_results[model])
        print("-" * 40)
    except Exception as e:
        audit_results[model] = f"ERROR: {str(e)}"
        print(f"✗ Failed: {e}")


# --- Save ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"dow_study_{timestamp}.json"

output = {
    "timestamp": timestamp,
    "prompt": GENERATION_PROMPT,
    "audit_prompt": AUDIT_PROMPT,
    "models": models,
    "generated": results,
    "audits": audit_results
}

with open(filename, "w") as f:
    json.dump(output, f, indent=2)

print("\n")
print("=" * 60)
print(f"COMPLETE — saved to {filename}")
print("=" * 60)

# --- Summary ---
print("\nSUMMARY")
print("-" * 60)
for model in models:
    gen = results.get(model, "")
    audit = audit_results.get(model, "")
    gen_status = "✗ ERROR" if gen.startswith("ERROR") else "✓ Generated"
    audit_status = "✗ ERROR" if audit.startswith("ERROR") else "✓ Audited"
    print(f"{model:<45} {gen_status}  {audit_status}")